In [1]:
!pip install pandas numpy scikit-learn tensorflow matplotlib yfinance

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------ --------------------------- 0.5/1.7 MB 5.1 MB/s eta 0:00:01
   ------------ --------------------------- 0.5/1.7 MB 5.1 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 2.7 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN
import matplotlib.pyplot as plt
import yfinance as yf


In [4]:
# List of festivals with their dates
# festivals that have already occurred in 2026 (as of April 19)
festivals = [
    {'name' : 'Makar Sankranti', 'date' : '2026-01-14'},
    {'name' : 'Vasant Panchami', 'date' : '2026-02-02'},
    {'name' : 'Maha Shivratri', 'date' : '2026-02-15'},
    {'name' : 'Holika Dahan', 'date' : '2026-03-03'},
    {'name' : 'Holi', 'date' : '2026-03-04'},
    {'name' : 'Eid al-Fitr', 'date' : '2026-03-30'},
    {'name' : 'Good Friday', 'date' : '2026-04-03'},
    {'name' : 'Easter', 'date' : '2026-04-05'},
    {'name' : 'Ram Navami', 'date' : '2026-03-27'},
    {'name' : 'Baisakhi', 'date' : '2026-04-14'},
    {'name' : 'Vishu', 'date' : '2026-04-14'},
]

In [6]:
# Calculate date ranges for each festival
def get_date_range(festival_date):
    festival_date = datetime.strptime(festival_date,'%Y-%m-%d')
    start_date = festival_date - timedelta(days=10)
    end_date = festival_date + timedelta(days=10)
    return start_date, end_date


In [7]:
print(get_date_range('2026-05-01'))

(datetime.datetime(2026, 4, 21, 0, 0), datetime.datetime(2026, 5, 11, 0, 0))


In [8]:
# prepare date Range
date_ranges = [
    {
        'festival' : festival['name'],
        'festival_date' : festival['date'],
        'start_date' : get_date_range(festival['date'])[0],
        'end_date' : get_date_range(festival['date'])[1]
    }
    for festival in festivals
]

In [43]:
def fetch_and_train_model(date_range):
    print(
        f'Precessing festival: {date_range['festival']}, '
        f'Start Date: {date_range['start_date'].date()}, End Date: {date_range['end_date'].date()}'
    )

    try:
        # fetch stock data
        data = yf.download(
            'MANAPPURAM.NS', start=date_range['start_date'].date(), end=date_range['end_date'].date()
        )
        print(f'raw data')
        print(data)

        print('closing values')
        data = data[['Close']]
        print(data)

        # normalize the data
        scaler =MinMaxScaler(feature_range=(0, 1))
        scaled_data = scaler.fit_transform(data)
        print('after normalization')
        print(scaled_data)

        # prepare training data
        X = scaled_data[:-1] #all expect the last value as input
        y = scaled_data[1:] # all except the first value as target
        X = X.reshape(X.shape[0], 1, 1) # reshape to (samples, timesteps, features)

        #split into train and test sets
        train_size = int(len(X) * 0.8)
        x_train, x_test = X[:train_size], X[train_size:] # [0:11] --> Training, [11:end] -> Testing
        y_train, y_test = y[:train_size], y[train_size:]

        #build the RNN model using SimpleRNN
        model = Sequential([
            SimpleRNN(50, activation='relu', return_sequences=True, input_shape=(x_train.shape[1], 1)),
            SimpleRNN(50, activation='relu'), # second SimpleRNN layer
            Dense(1) #Output layer
        ])

        #Complie and train the model
        model.compile(loss='mean_squared_error', optimizer='adam')
        model.fit(x_train, y_train, epochs=50, batch_size=20, verbose=1)

        #predict
        predicted_prices = model.predict(x_test)
        predicted_prices = scaler.inverse_transform(predicted_prices.reshape(-1, 1))
        actual_prices = scaler.inverse_transform(y_test.reshape(-1, 1))
        return data.index[-len(y_test):], actual_prices, predicted_prices
    except Exception as ex:
        print(f'Error while processing {date_range['festival']} : {ex}')
        return [], [], []

In [44]:
fetch_and_train_model(date_ranges[0])

[*********************100%***********************]  1 of 1 completed

Precessing festival: Makar Sankranti, Start Date: 2026-01-04, End Date: 2026-01-24
raw data
Price              Close          High  ...          Open        Volume
Ticker     MANAPPURAM.NS MANAPPURAM.NS  ... MANAPPURAM.NS MANAPPURAM.NS
Date                                    ...                            
2026-01-05    306.093994    311.675655  ...    310.180569       3775651
2026-01-06    307.240204    309.383155  ...    308.436285       2069927
2026-01-07    318.852020    320.546463  ...    307.240191       8374754
2026-01-08    308.486084    319.151043  ...    318.852008       4155251
2026-01-09    284.913574    311.326761  ...    308.486092      28986822
2026-01-12    293.236237    297.721495  ...    289.947029      15984613
2026-01-13    307.040833    308.286738  ...    293.983772      12463499
2026-01-14    307.987732    308.585772  ...    305.994284       3052968
2026-01-15    307.987732    307.987732  ...    307.987732             0
2026-01-16    312.971344    315.712335  ... 


c:\Users\subramm6\Documents\Learn\gen-ai\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - loss: 0.4336
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - loss: 0.4099
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step - loss: 0.3876
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step - loss: 0.3665
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step - loss: 0.3461
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step - loss: 0.3265
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step - loss: 0.3076
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - loss: 0.2894
Epoch 9/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - loss: 0.2719
Epoch 10/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - loss: 0.2553
Epoch 11/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - loss: 0.2395
Epoch 12/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - loss: 0.2246
Epoch 13/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 0.2103
Epoch 14/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - loss: 0.1964
Epoch 15/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - loss: 0.1832
Epoch 16/50
1/1 ━━━━━━━━━━━━━━━

(DatetimeIndex(['2026-01-21', '2026-01-22', '2026-01-23'], dtype='datetime64[s]', name='Date', freq=None),
 array([[297.57196045],
        [299.21658325],
        [293.78442383]]),
 array([[303.47797],
        [301.4676 ],
        [302.47308]], dtype=float32))